In [1]:
# ============================================================
# EMBEDDINGS SEMÂNTICOS E BUSCA VETORIAL COM FAISS
# ============================================================

import sys
import os

In [2]:
# Adiciona a raiz do projeto ao sys.path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
print("✅ Caminho adicionado:", sys.path[0])

# ============================================================
# 1. Importações
# ============================================================
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

✅ Caminho adicionado: d:\3.1.pos_e_graduacao\INFNET\modulos\sistemas_cognitivos\sistemas_cognitivos_com_llms\pd-sistemas-cognitivos


d:\3.1.pos_e_graduacao\INFNET\modulos\sistemas_cognitivos\sistemas_cognitivos_com_llms\pd-sistemas-cognitivos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 2. Carregar Modelo de Embeddings
# ============================================================
print("\n" + "="*50)
print("Carregando modelo de embeddings...")
print("="*50)

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ Modelo carregado. Dimensão do embedding: {model.get_sentence_embedding_dimension()}")

# ============================================================
# 3. Criar Corpus de Exemplo (Artigos Científicos)
# ============================================================
docs = [
    "Deep learning é uma subárea de machine learning que utiliza redes neurais com múltiplas camadas.",
    "Processamento de linguagem natural (PLN) lida com a compreensão e geração de linguagem humana por computadores.",
    "Aprendizado por reforço é usado em robótica e jogos, onde agentes aprendem por tentativa e erro.",
    "Visão computacional permite que máquinas interpretem e entendam imagens e vídeos.",
    "Transformers revolucionaram o PLN com mecanismos de atenção, como no BERT e GPT."
]
print(f"\n📚 Corpus com {len(docs)} documentos.")

# ============================================================
# 4. Gerar Embeddings
# ============================================================
print("\n" + "="*50)
print("Gerando embeddings...")
print("="*50)

vectors = model.encode(docs, convert_to_numpy=True, show_progress_bar=True)
print(f"✅ Embeddings gerados: {vectors.shape}")

# Normaliza para similaridade por cosseno
faiss.normalize_L2(vectors)

# ============================================================
# 5. Indexar com FAISS
# ============================================================
dimension = vectors.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product = cosseno (vetores normalizados)
index.add(vectors)
print(f"✅ Índice FAISS criado com {index.ntotal} vetores.")

# ============================================================
# 6. Buscar Documentos Similares
# ============================================================
print("\n" + "="*50)
print("Buscando documentos similares...")
print("="*50)

query = "Como funcionam as redes neurais?"
query_vec = model.encode([query], convert_to_numpy=True)
faiss.normalize_L2(query_vec)

k = 3  # top-k
distances, indices = index.search(query_vec, k)

print(f"\n🔍 Consulta: '{query}'\n")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"{i+1}. Score: {dist:.4f}")
    print(f"   Documento: {docs[idx]}")
    print()

# ============================================================
# 7. Busca com Múltiplas Consultas
# ============================================================
print("\n" + "="*50)
print("Teste com múltiplas consultas:")
print("="*50)

queries = [
    "robótica e aprendizado por reforço",
    "compreensão de linguagem",
    "reconhecimento de imagens"
]

for q in queries:
    print(f"\n🔍 '{q}':")
    q_vec = model.encode([q], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    distances, indices = index.search(q_vec, 2)
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        print(f"   {i+1}. {docs[idx]} (score: {dist:.4f})")

# ============================================================
# 8. Discussão dos Resultados
# ============================================================
print("\n" + "="*50)
print("ANÁLISE DA RECUPERAÇÃO")
print("="*50)
print("""
✅ A busca por similaridade por cosseno recuperou documentos relevantes.
✅ Os scores indicam a força da similaridade semântica.
⚠️ Consultas muito específicas podem não encontrar correspondência exata.
💡 Para melhorar, pode-se usar busca híbrida (BM25 + vetorial).
""")

print("\n✅ Notebook executado com sucesso!")


Carregando modelo de embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6818.71it/s]
C:\Users\walla\AppData\Local\Temp\ipykernel_18916\2708754003.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Modelo carregado. Dimensão do embedding: {model.get_sentence_embedding_dimension()}")


✅ Modelo carregado. Dimensão do embedding: 384

📚 Corpus com 5 documentos.

Gerando embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.40it/s]

✅ Embeddings gerados: (5, 384)
✅ Índice FAISS criado com 5 vetores.

Buscando documentos similares...

🔍 Consulta: 'Como funcionam as redes neurais?'

1. Score: 0.4187
   Documento: Deep learning é uma subárea de machine learning que utiliza redes neurais com múltiplas camadas.

2. Score: 0.3964
   Documento: Transformers revolucionaram o PLN com mecanismos de atenção, como no BERT e GPT.

3. Score: 0.3519
   Documento: Visão computacional permite que máquinas interpretem e entendam imagens e vídeos.


Teste com múltiplas consultas:

🔍 'robótica e aprendizado por reforço':
   1. Aprendizado por reforço é usado em robótica e jogos, onde agentes aprendem por tentativa e erro. (score: 0.8609)
   2. Visão computacional permite que máquinas interpretem e entendam imagens e vídeos. (score: 0.4233)

🔍 'compreensão de linguagem':
   1. Processamento de linguagem natural (PLN) lida com a compreensão e geração de linguagem humana por computadores. (score: 0.6109)
   2. Transformers revolucionara